In [1]:
print("hi")

hi


In [2]:
from langchain_community.document_loaders import TextLoader, Docx2txtLoader, UnstructuredWordDocumentLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


loader = TextLoader("report.txt", encoding="utf-8")
documents = loader.load()
# print(documents)
# print(documents[0].page_content)

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 80
)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vb = FAISS.from_documents(
    docs,
    embeddings
)

vb.save_local("vectorstore")
print("VectorStore created")

# print(vb)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4910.14it/s]


VectorStore created


In [3]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict

from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

veb = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization = True
)

ret = veb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 20
    }
)

llm = ChatGroq(
    model = "groq/compound-mini",
    temperature = 0
)

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
# res = llm.invoke("what can you do?")
# res.content

d:\AI_dev\CASO\caso-lib\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
d:\AI_dev\CASO\caso-lib\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\asus\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows

In [4]:
def rerank_docs(query, docs):

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in ranked_docs[:4]
    ]

In [12]:
class RagState(TypedDict):
    
    question : str
    context : str
    answer : str
    
def retrieve(state):

    question = state["question"]

    docs = ret.invoke(question)

    docs = rerank_docs(question, docs)
    
    for i, doc in enumerate(docs):
        print(f"\n--- DOC {i+1} ---")
        print(doc.page_content)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return {
        "context": context
    }
    
def generator(state: RagState):
    
    ques = state.get("question", "")
    con = state.get("context", "")
    
    prompt = f"""
You are an enterprise AI assistant.

Rules:
- Answer ONLY from the provided context
- If answer is missing, say:
  "I could not find this information in the context."
- Do not hallucinate
- Be concise and factual

Context:
{con}

Question:
{ques}
"""
    res = llm.invoke(prompt)
    
    return {
        "answer" : res.content
    }

In [13]:
from pprint import pprint

sg = StateGraph(RagState)

sg.add_node("retriver", retrieve)
# sg.add_node("reranker", rerank_docs)
sg.add_node("generator", generator)

# sg.set_entry_point("retriver")
sg.add_edge(START,"retriver")
# sg.add_edge("retriver", "reranker")
sg.add_edge("retriver", "generator")
sg.add_edge("generator", END)

graph = sg.compile()

response = graph.invoke({
    "question": "summarize chapter 1"
})

print("\nANSWER:\n")
pprint(response["answer"])


--- DOC 1 ---
List of Symbols and Acronyms.....................................................................................vii
Chapter 1: Introduction...............................................................................................1

--- DOC 2 ---
1.5 Report Structure......................................................................................................3
Chapter 2: Literature Review / Background Work ...................................................4

--- DOC 3 ---
What comes next breaks down like this. After looking at current legal AI setups, Chapter 2 points out what they miss - showing where this effort steps in. Instead of just listing parts, Chapter 3 walks through how the system is built, including methods used and visual maps of data movement. Moving

--- DOC 4 ---
dealt with workers fired without reason. Every example came written out like a person might explain their problem in everyday words. These were not scripted summaries but full litt